# CIC-IDS2017 — Dataset Exploration

CICFlowMeter-generated network flow features across one working week, covering DoS, DDoS, brute-force, port scan, web attacks, botnet, and infiltration.

**Known issues:** leading whitespace in column names, `Inf` values in flow rate columns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
BASE = Path('../../dataset/cicids2017/')

## 1 · Load & Merge All Day Files

In [ ]:
files = sorted(BASE.glob('*.csv'))
print('Files:')
for f in files: print(f'  {f.name}')

dfs = []
for f in files:
    tmp = pd.read_csv(f, low_memory=False)
    tmp['source_file'] = f.stem
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)

# Strip leading/trailing whitespace from column names (known issue)
df.columns = df.columns.str.strip()
print(f'\nCombined shape: {df.shape}')
df.head(3)

## 2 · Inf and NaN Values

CICFlowMeter produces `Inf` in flow-rate columns when duration is zero.

In [ ]:
inf_counts = (df == np.inf).sum()
nan_counts = df.isnull().sum()

print('Columns with Inf values:')
print(inf_counts[inf_counts > 0].to_string())
print()
print('Columns with NaN values:')
print(nan_counts[nan_counts > 0].to_string() or 'None')

## 3 · Label Distribution

In [ ]:
print('Label value counts:')
print(df['Label'].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
# Multi-class
df['Label'].value_counts().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Multi-class label counts (log scale)')
axes[0].set_xscale('log')
# Binary
df['Label'].apply(lambda x: 'BENIGN' if x == 'BENIGN' else 'ATTACK') \
    .value_counts().plot(kind='bar', ax=axes[1], color=['steelblue','tomato'])
axes[1].set_title('Binary split'); axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 4 · Labels per Day

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df.groupby('source_file')['Label'].value_counts().unstack(fill_value=0) \
    .plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Attack types per capture day')
ax.tick_params(axis='x', rotation=20)
ax.legend(loc='upper right', fontsize=7)
plt.tight_layout(); plt.show()

## 5 · Feature Statistics (excluding Inf)

In [ ]:
num_cols = df.select_dtypes('number').columns
# Replace Inf with NaN for statistics
df_stats = df[num_cols].replace([np.inf, -np.inf], np.nan)
df_stats.describe().T.round(3)

## 6 · Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(15, 13))
corr = df_stats.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            linewidths=0.2, ax=ax, cbar_kws={'shrink': 0.7},
            xticklabels=True, yticklabels=True)
ax.set_title('Correlation matrix')
plt.tight_layout(); plt.show()

## 7 · Preprocessing Checklist

| Step | Required | Notes |
|---|---|---|
| Strip column name whitespace | **Done above** | Apply consistently |
| Replace Inf with NaN | Yes | `Flow Bytes/s`, `Flow Packets/s` |
| Drop or impute NaN | Yes | After Inf replacement |
| Drop `source_file` column | Yes | Metadata, not a feature |
| Encode labels to integers | Yes | 15 classes or binary |
| MinMax / Standard scale | Yes | Wide value ranges across features |
| Near-zero variance features | Check | Some flag-count columns may be constant |
| Class imbalance | Severe | BENIGN ~80 %, many attacks < 1 % |

## 8 · Train / Test Split

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Drop rows with Inf (alternative: clip/impute)
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=num_cols)
df_clean = df_clean.drop(columns=['source_file'])

le = LabelEncoder()
y = le.fit_transform(df_clean['Label'])
X = df_clean.drop(columns=['Label'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Classes: {list(le.classes_)}')

## 9 · Summary

- **Size:** ~2.8 M rows (across 8 day files)
- **Features:** 78 numeric (CICFlowMeter)
- **Known bugs:** Leading whitespace in headers, `Inf` in rate columns
- **Labels:** 15 classes including BENIGN; highly imbalanced
- **Key preprocessing:** strip headers, replace Inf→NaN, drop/impute, scale